# Phase 3 Error taxonomy

Goal: manually assign an error code to the 30 explanations with the lowest combined
RA/SA score (Ichmoukhamedov metric), describe the categories and count their frequency.

Note: this notebook is the frozen diagnostic of the initial run that motivated the
prompt revision. The manual coding below is a snapshot of those explanations.

Method: for each case we compare:
1. Ground truth: local SHAP/EBM contributions from explanations/local_*.json (sorted by absolute contribution)
2. Generated explanation: text from results/pipeline0{4,5,6}/*.json
3. Extraction: structured JSON from results/eval06_ichmoukhamedov/extractions.json
   (what notebook 06 read out of the explanation)

The error source can be either the explanation itself or the LLM based extractor in NB 06.

## 1. Taxonomy definitions

| Code | Name | Description | Source |
|------|------|-------------|--------|
| E1 | Extractor feature swap | The NB 06 extractor assigns the wrong feature as rank 0, although the explanation names the correct feature. | Eval |
| E2 | Extractor sign inversion | The extractor identifies the correct feature at the correct rank but inverts the sign (positive contribution to sign=-1 or the other way around). | Eval |
| E3 | Extractor cross instance hallucination | The extractor pulls context from a completely different scenario (different hour, weekday, year) than the present instance. | Eval |
| B1 | Positive/negative grouping | The explanation sorts features by sign first (all drivers, then all brakes) instead of by absolute contribution. The extractor adopts this order, which deviates from the SHAP ranking. | Explanation |
| B2 | Close contribution rank swap | Two features with very similar absolute contribution are named in reverse order in the explanation or by the extractor. | Explanation/Eval |
| C | Sign error yr/temp | The explanation gives the wrong sign for the year or temperature feature. Typical: yr=0 (2011, negative contribution) is shown as positive "2012 growth". | Explanation |

Note: categories E1 to E3 are artefacts of the evaluator (NB 06), not errors of the XAI pipeline.
Categories B1, B2 and C are errors in the generated explanation.

In [ ]:
import os
import sys
import json
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

from utils import EXPLANATIONS_DIR, RESULTS_DIR
from utils.faithfulness import (
    rank0_correctness, correct_metric, load_explanation_text, load_gt_contributions,
)
from IPython.display import display

# Paths via the project constants (utils); BASE only for the few os.path calls.
BASE = str(Path.cwd().parent)
print('PROJECT_ROOT:', BASE)

In [ ]:
# --- Manual coding of the 30 worst-RA/SA cases ---
# Selection: bottom-30 by avg(RA,SA), excluding Template pipeline (no LLM).
# Each case was examined by comparing:
#   (a) local SHAP/EBM contributions (ground truth rank & sign),
#   (b) the generated explanation text,
#   (c) the structured extraction produced by NB 06.

CODED_CASES = [
    # ---- WORST (RA=0, SA=0) ----
    dict(case_key='05_xgb_inst1041', pipeline='Vision',    xai='XGB', instance=1041,
         RA=0.00, SA=0.00, VA=0.50,
         primary='E3', secondary='E1,E2',
         gt_rank0_feature='hr',      gt_rank0_sign=+1, gt_rank0_contrib=0.747,
         ext_rank0_feature='weekday', ext_rank0_sign=-1,
         note='Extractor creates Tuesday-11h context; actual instance is Friday-8h (hr=8 is rank-0 by SHAP). '
              'Explanation correctly states hr=8h as strongest driver.'),

    dict(case_key='05_ebm_inst1481', pipeline='Vision',    xai='EBM', instance=1481,
         RA=0.00, SA=0.00, VA=0.67,
         primary='E3', secondary='',
         gt_rank0_feature='hr',           gt_rank0_sign=+1, gt_rank0_contrib=0.819,
         ext_rank0_feature='hr_weekday',   ext_rank0_sign=+1,
         note='Extractor hallucinates a Friday-23h scenario (hr=23, yr=1=2012). '
              'Actual instance: Tuesday-16h, yr=0=2011. Explanation is correct.'),

    dict(case_key='06_xgb_inst1041', pipeline='Tool Use',  xai='XGB', instance=1041,
         RA=0.00, SA=0.00, VA=0.50,
         primary='E3', secondary='E1,E2',
         gt_rank0_feature='hr',      gt_rank0_sign=+1, gt_rank0_contrib=0.747,
         ext_rank0_feature='weekday', ext_rank0_sign=-1,
         note='Same extractor hallucination as 05_xgb_inst1041. '
              'Tool-Use explanation correctly identifies hr as top driver.'),

    dict(case_key='04_xgb_inst1041', pipeline='JSON to Text', xai='XGB', instance=1041,
         RA=0.00, SA=0.33, VA=0.67,
         primary='E3', secondary='E1,E2',
         gt_rank0_feature='hr',      gt_rank0_sign=+1, gt_rank0_contrib=0.747,
         ext_rank0_feature='weekday', ext_rank0_sign=-1,
         note='Same extractor hallucination (Tuesday value=2 invented). '
              'JSON-Text explanation correctly starts with hr=8h Morgenspitze.'),

    # ---- E2: Sign inversion at rank 0 ----
    dict(case_key='04_xgb_inst1481', pipeline='JSON to Text', xai='XGB', instance=1481,
         RA=0.50, SA=0.00, VA=0.50,
         primary='E2', secondary='',
         gt_rank0_feature='hr', gt_rank0_sign=+1, gt_rank0_contrib=0.378,
         ext_rank0_feature='hr', ext_rank0_sign=-1,
         note='hr correctly at rank 0, but extractor inverts sign (+\u2192-). '
              'All three pipeline explanations correctly describe hr=16h as upward driver.'),

    # ---- E1: Feature swap at rank 0 ----
    dict(case_key='05_xgb_inst224',  pipeline='Vision',    xai='XGB', instance=224,
         RA=0.00, SA=0.50, VA=0.50,
         primary='E1', secondary='E2',
         gt_rank0_feature='temp', gt_rank0_sign=-1, gt_rank0_contrib=0.549,
         ext_rank0_feature='hr',  ext_rank0_sign=+1,
         note='Explanation correctly says temp(~8\xb0C) is the dominant downward driver. '
              'Extractor assigns hr(+) as rank 0, inverting both feature and sign.'),

    dict(case_key='05_ebm_inst580',  pipeline='Vision',    xai='EBM', instance=580,
         RA=0.00, SA=0.67, VA=0.33,
         primary='E1', secondary='',
         gt_rank0_feature='yr', gt_rank0_sign=-1, gt_rank0_contrib=0.226,
         ext_rank0_feature='hr', ext_rank0_sign=-1,
         note='yr(-) is rank-0 by SHAP. Extractor assigns hr(-) as rank 0. '
              'Sign is correct for hr, but it should be yr at rank 0.'),

    dict(case_key='06_xgb_inst224',  pipeline='Tool Use',  xai='XGB', instance=224,
         RA=0.00, SA=0.67, VA=1.00,
         primary='E1', secondary='E2',
         gt_rank0_feature='temp', gt_rank0_sign=-1, gt_rank0_contrib=0.549,
         ext_rank0_feature='hr',  ext_rank0_sign=+1,
         note='Same as 05_xgb_inst224. Tool-Use explanation correctly leads with temp as brake.'),

    dict(case_key='04_xgb_inst224',  pipeline='JSON to Text', xai='XGB', instance=224,
         RA=0.50, SA=0.50, VA=0.50,
         primary='E1', secondary='E2',
         gt_rank0_feature='temp', gt_rank0_sign=-1, gt_rank0_contrib=0.549,
         ext_rank0_feature='hr',  ext_rank0_sign=+1,
         note='Same extractor error: hr(+) extracted at rank 0 instead of temp(-). '
              'Explanation correctly names temp as strongest brake.'),

    dict(case_key='04_ebm_inst580',  pipeline='JSON to Text', xai='EBM', instance=580,
         RA=0.25, SA=0.75, VA=0.67,
         primary='E1', secondary='',
         gt_rank0_feature='yr', gt_rank0_sign=-1, gt_rank0_contrib=0.226,
         ext_rank0_feature='hr', ext_rank0_sign=-1,
         note='Same as 05_ebm_inst580. Extractor puts hr at rank 0 instead of yr.'),

    # ---- C: yr sign error in explanation ----
    dict(case_key='04_ebm_inst1041', pipeline='JSON to Text', xai='EBM', instance=1041,
         RA=0.25, SA=0.75, VA=0.67,
         primary='C', secondary='B2',
         gt_rank0_feature='hr', gt_rank0_sign=+1, gt_rank0_contrib=1.109,
         ext_rank0_feature='hr', ext_rank0_sign=+1,
         note='hr rank-0 correct. yr has contribution=-0.226 (negative: instance yr=0=2011), '
              'but explanation frames yr as "2012 is growing" with positive sign. '
              'Causes SA miss for yr. temp/yr ranks also swapped (B2).'),

    # ---- B2: Close-contribution rank swap ----
    dict(case_key='04_ebm_inst2058', pipeline='JSON to Text', xai='EBM', instance=2058,
         RA=0.25, SA=0.75, VA=0.75,
         primary='B2', secondary='E2',
         gt_rank0_feature='hr',   gt_rank0_sign=+1, gt_rank0_contrib=0.555,
         ext_rank0_feature='hr',  ext_rank0_sign=+1,
         note='hr rank-0 correct. temp(0.230) and yr(0.226) differ by <2% \u2014 '
              'extractor swaps them (yr\u2192rank1, temp\u2192rank3). '
              'mnth sign also inverted by extractor (GT=+0.112, EXT=sign-1).'),

    dict(case_key='05_xgb_inst1481', pipeline='Vision',    xai='XGB', instance=1481,
         RA=1.00, SA=0.00, VA=0.33,
         primary='E2', secondary='',
         gt_rank0_feature='hr', gt_rank0_sign=+1, gt_rank0_contrib=0.378,
         ext_rank0_feature='hr', ext_rank0_sign=-1,
         note='Rank fully correct (RA=1). Extractor inverts ALL signs: '
              'hr(+\u2192-), yr(-\u2192+), all other features similarly flipped. '
              'Explanation correctly identifies hr=16h as upward driver.'),

    # ---- B1: Positive/Negative grouping ----
    dict(case_key='05_xgb_inst1677', pipeline='Vision',    xai='XGB', instance=1677,
         RA=0.00, SA=1.00, VA=0.67,
         primary='B1', secondary='E1',
         gt_rank0_feature='hr', gt_rank0_sign=+1, gt_rank0_contrib=0.253,
         ext_rank0_feature='yr', ext_rank0_sign=-1,
         note='hr(+0.253) and yr(-0.235) are nearly equal in absolute value. '
              'Explanation groups: "hr & temp are positive drivers, yr is the biggest brake". '
              'Extractor interprets "biggest brake" as rank-0, but SHAP rank-0 is hr by absolute value. '
              'All signs correct (SA=1).'),

    dict(case_key='05_xgb_inst3543', pipeline='Vision',    xai='XGB', instance=3543,
         RA=0.00, SA=1.00, VA=0.50,
         primary='E1', secondary='B2',
         gt_rank0_feature='yr',  gt_rank0_sign=+1, gt_rank0_contrib=0.186,
         ext_rank0_feature='hr', ext_rank0_sign=+1,
         note='yr(+0.186) is rank-0, hr(+0.107) is rank-1. '
              'Extractor assigns hr as rank-0. '
              'Vision explanation presumably led with hr (Uhrzeit 20h). '
              'Contributions not especially close \u2014 likely extractor reading explanation order.'),

    dict(case_key='05_xgb_inst4454', pipeline='Vision',    xai='XGB', instance=4454,
         RA=0.25, SA=0.75, VA=0.50,
         primary='B2', secondary='E2',
         gt_rank0_feature='hr',   gt_rank0_sign=+1, gt_rank0_contrib=0.352,
         ext_rank0_feature='hr',  ext_rank0_sign=+1,
         note='hr rank-0 correct. yr(0.145) and temp(0.158) at ranks 2/3 are very close (<10% apart) '
              'and are swapped. weekday sign inverted by extractor (GT=-0.108, EXT=+1).'),

    dict(case_key='05_ebm_inst1041', pipeline='Vision',    xai='EBM', instance=1041,
         RA=0.33, SA=0.67, VA=0.33,
         primary='C', secondary='B2',
         gt_rank0_feature='hr', gt_rank0_sign=+1, gt_rank0_contrib=1.109,
         ext_rank0_feature='hr', ext_rank0_sign=+1,
         note='hr rank-0 correct. yr(contribution=-0.226) again framed as positive "2012-Wachstum" '
              '\u2192 SA miss on yr. temp(0.098) presented before yr(0.226) despite lower |contribution| (B2).'),

    dict(case_key='05_ebm_inst2510', pipeline='Vision',    xai='EBM', instance=2510,
         RA=0.25, SA=0.75, VA=0.75,
         primary='C', secondary='B2',
         gt_rank0_feature='hr',   gt_rank0_sign=+1, gt_rank0_contrib=1.148,
         ext_rank0_feature='hr',  ext_rank0_sign=+1,
         note='hr rank-0 correct. temp has GT contribution=-0.062 (negative) but extracted as sign=+1. '
              'yr(0.226) and temp(0.062) rank-swapped. weekday injected at rank-1 (not in GT top-3).'),

    dict(case_key='06_xgb_inst1481', pipeline='Tool Use',  xai='XGB', instance=1481,
         RA=1.00, SA=0.00, VA=0.50,
         primary='E2', secondary='',
         gt_rank0_feature='hr', gt_rank0_sign=+1, gt_rank0_contrib=0.378,
         ext_rank0_feature='hr', ext_rank0_sign=-1,
         note='Same extractor sign-inversion as 05_xgb_inst1481 (RA=1, SA=0). '
              'Tool-Use explanation correctly describes hr=16h as strong upward driver.'),

    dict(case_key='06_xgb_inst1677', pipeline='Tool Use',  xai='XGB', instance=1677,
         RA=0.00, SA=1.00, VA=0.67,
         primary='B1', secondary='E1',
         gt_rank0_feature='hr', gt_rank0_sign=+1, gt_rank0_contrib=0.253,
         ext_rank0_feature='yr', ext_rank0_sign=-1,
         note='Same positive/negative-grouping error as 05_xgb_inst1677. '
              'Tool-Use explanation leads with hr(+) then yr(-), '
              'extractor promotes yr to rank-0 because it is called "biggest brake".'),

    dict(case_key='06_xgb_inst3543', pipeline='Tool Use',  xai='XGB', instance=3543,
         RA=0.00, SA=1.00, VA=0.67,
         primary='E1', secondary='B2',
         gt_rank0_feature='yr',  gt_rank0_sign=+1, gt_rank0_contrib=0.186,
         ext_rank0_feature='hr', ext_rank0_sign=+1,
         note='Tool-Use explanation correctly identifies yr as rank-0 ("strongest positive influence: year 2012"). '
              'But extractor assigns hr as rank-0 \u2014 clear extractor error.'),

    dict(case_key='06_xgb_inst3847', pipeline='Tool Use',  xai='XGB', instance=3847,
         RA=0.33, SA=0.67, VA=0.67,
         primary='E2', secondary='',
         gt_rank0_feature='hr', gt_rank0_sign=+1, gt_rank0_contrib=0.413,
         ext_rank0_feature='hr', ext_rank0_sign=-1,
         note='hr rank-0 correct. Extractor inverts hr sign (+\u2192-). '
              'Lower ranks also partially wrong.'),

    dict(case_key='06_ebm_inst580',  pipeline='Tool Use',  xai='EBM', instance=580,
         RA=0.33, SA=0.67, VA=0.67,
         primary='E1', secondary='',
         gt_rank0_feature='yr', gt_rank0_sign=-1, gt_rank0_contrib=0.226,
         ext_rank0_feature='hr', ext_rank0_sign=-1,
         note='yr(-) is SHAP rank-0. Extractor assigns hr(-) as rank-0. '
              'Sign is correct for hr but feature identity is wrong.'),

    dict(case_key='06_ebm_inst1041', pipeline='Tool Use',  xai='EBM', instance=1041,
         RA=0.33, SA=0.67, VA=0.67,
         primary='C', secondary='B2',
         gt_rank0_feature='hr', gt_rank0_sign=+1, gt_rank0_contrib=1.109,
         ext_rank0_feature='hr', ext_rank0_sign=+1,
         note='hr rank-0 correct. Same yr sign error as 04/05_ebm_inst1041: '
              'yr=0(2011) has negative contribution but Tool-Use explanation presents "2012 growing".'),

    dict(case_key='06_ebm_inst2510', pipeline='Tool Use',  xai='EBM', instance=2510,
         RA=0.33, SA=0.67, VA=1.00,
         primary='C', secondary='B2',
         gt_rank0_feature='hr',  gt_rank0_sign=+1, gt_rank0_contrib=1.148,
         ext_rank0_feature='hr', ext_rank0_sign=+1,
         note='hr rank-0 correct. temp extracted with wrong sign (GT=-0.062, EXT=+1). '
              'yr and temp rank swapped. Same pattern as 05_ebm_inst2510.'),

    dict(case_key='04_xgb_inst3543', pipeline='JSON to Text', xai='XGB', instance=3543,
         RA=0.25, SA=1.00, VA=0.75,
         primary='E1', secondary='B2',
         gt_rank0_feature='yr',  gt_rank0_sign=+1, gt_rank0_contrib=0.186,
         ext_rank0_feature='hr', ext_rank0_sign=+1,
         note='JSON-Text explanation correctly opens with "year 2012 is the strongest driver". '
              'Extractor assigns hr as rank-0 \u2014 clear extractor error. All signs correct (SA=1).'),

    dict(case_key='04_xgb_inst3847', pipeline='JSON to Text', xai='XGB', instance=3847,
         RA=0.50, SA=0.75, VA=0.75,
         primary='E2', secondary='',
         gt_rank0_feature='hr', gt_rank0_sign=+1, gt_rank0_contrib=0.413,
         ext_rank0_feature='hr', ext_rank0_sign=-1,
         note='hr rank-0 correct. Extractor inverts hr sign (+\u2192-). Partial rank matches.'),

    dict(case_key='04_ebm_inst3847', pipeline='JSON to Text', xai='EBM', instance=3847,
         RA=0.75, SA=0.50, VA=0.50,
         primary='E2', secondary='',
         gt_rank0_feature='hr', gt_rank0_sign=+1, gt_rank0_contrib=0.661,
         ext_rank0_feature='hr', ext_rank0_sign=-1,
         note='hr rank-0 correct (RA=0.75 \u2014 best rank match in this set). '
              'Extractor inverts hr sign. Mostly a measurement artifact.'),

    dict(case_key='05_xgb_inst580',  pipeline='Vision',    xai='XGB', instance=580,
         RA=0.25, SA=1.00, VA=0.33,
         primary='B2', secondary='',
         gt_rank0_feature='hr',  gt_rank0_sign=-1, gt_rank0_contrib=0.365,
         ext_rank0_feature='hr', ext_rank0_sign=-1,
         note='hr rank-0 correct. All four features (hr, yr, mnth, temp) are negative \u2014 '
              'all contributions between 0.22 and 0.36. '
              'yr(0.328)/mnth(0.248)/temp(0.220) rank-swapped in extraction: '
              'mnth appears before yr despite lower |contribution|. All signs correct (SA=1).'),

    dict(case_key='05_xgb_inst2058', pipeline='Vision',    xai='XGB', instance=2058,
         RA=0.50, SA=0.75, VA=0.67,
         primary='B2', secondary='E1',
         gt_rank0_feature='yr',  gt_rank0_sign=-1, gt_rank0_contrib=0.195,
         ext_rank0_feature='yr', ext_rank0_sign=-1,
         note='yr rank-0 correct. hr (not in GT top-4) injected at rank-1. '
              'temp pushed to rank-7 despite being GT rank-1 (+0.163). '
              'mnth sign inverted (GT=+0.123, EXT=sign-1).'),
]

df_coded = pd.DataFrame(CODED_CASES)
print(f'Coded cases: {len(df_coded)}')
df_coded[['case_key','pipeline','xai','instance','RA','SA','VA','primary','secondary']].to_string(index=False)

## 2. Frequency count

In [ ]:
# Primary category counts
cat_counts = df_coded['primary'].value_counts().reset_index()
cat_counts.columns = ['Category', 'Count']

# Category labels
cat_labels = {
    'E1': 'E1 Extractor: feature swap',
    'E2': 'E2 Extractor: sign inversion',
    'E3': 'E3 Extractor: cross instance hallucination',
    'B1': 'B1 Explanation: positive/negative grouping',
    'B2': 'B2 Explanation: close contribution rank swap',
    'C':  'C  Explanation: sign error yr/temp',
}
cat_counts['Description'] = cat_counts['Category'].map(cat_labels)

# Group: eval artifact vs explanation error
cat_counts['Source'] = cat_counts['Category'].apply(
    lambda c: 'Eval artifact (NB 06)' if c.startswith('E') else 'Explanation error'
)

print(cat_counts[['Category','Description','Count','Source']].to_string(index=False))
print()
print('Sum eval artifacts:', cat_counts[cat_counts['Source']=='Eval artifact (NB 06)']['Count'].sum())
print('Sum explanation errors:', cat_counts[cat_counts['Source']=='Explanation error']['Count'].sum())

In [ ]:
# Save coded cases and frequency table
out_dir = os.path.join(BASE, 'results', 'error_taxonomy')
os.makedirs(out_dir, exist_ok=True)

df_coded.to_csv(os.path.join(out_dir, 'coded_cases.csv'), index=False)
cat_counts.to_csv(os.path.join(out_dir, 'category_frequencies.csv'), index=False)
print('Saved to', out_dir)

## 3. Visualisation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Panel A: frequency by category
ax = axes[0]
color_map = {
    'E1': '#d62728', 'E2': '#ff7f0e', 'E3': '#9467bd',
    'B1': '#1f77b4', 'B2': '#17becf', 'C': '#2ca02c',
}
order = ['E3', 'E2', 'E1', 'B1', 'B2', 'C']
counts_ordered = [df_coded['primary'].value_counts().get(c, 0) for c in order]
colors = [color_map[c] for c in order]

bars = ax.barh(order, counts_ordered, color=colors, edgecolor='white', height=0.6)
for bar, cnt in zip(bars, counts_ordered):
    ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
            str(cnt), va='center', fontsize=11, fontweight='bold')

ax.set_xlabel('Number of cases (of 30)', fontsize=11)
ax.set_title('A Primary category', fontsize=12, fontweight='bold')
ax.set_xlim(0, 11)
ax.invert_yaxis()

# Add legend for source
eval_patch = mpatches.Patch(facecolor='#d62728', alpha=0.7, label='Eval artifact (NB 06)')
expl_patch = mpatches.Patch(facecolor='#1f77b4', alpha=0.7, label='Explanation error')
ax.legend(handles=[eval_patch, expl_patch], loc='lower right', fontsize=9)

# Panel B: source split (pie)
ax2 = axes[1]
source_counts = cat_counts.groupby('Source')['Count'].sum()
wedges, texts, autotexts = ax2.pie(
    source_counts.values,
    labels=source_counts.index,
    autopct='%1.0f%%',
    colors=['#e08080', '#80b0e0'],
    startangle=90,
    textprops={'fontsize': 11},
)
for at in autotexts:
    at.set_fontsize(13)
    at.set_fontweight('bold')
ax2.set_title('B Error source (n=30)', fontsize=12, fontweight='bold')

plt.suptitle('Error taxonomy: 30 cases with the lowest RA/SA', fontsize=13, y=1.01)
plt.tight_layout()

fig_path = os.path.join(out_dir, 'taxonomy_overview.png')
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
display(fig)
print('Saved:', fig_path)

## 4. Example cases per category

In [ ]:
# Load raw artifacts for illustration
with open(os.path.join(BASE, 'results', 'eval06_ichmoukhamedov', 'extractions.json')) as f:
    extractions = json.load(f)

prefix_map = {'JSON to Text': '04', 'Vision': '05', 'Tool Use': '06'}


def show_case(case_key, label):
    """Display helper: ground truth, extraction and explanation of a case side by side.
    Loaders (SHAP GT, explanation text) come from utils.faithfulness."""
    row = df_coded[df_coded['case_key'] == case_key].iloc[0]
    prefix = prefix_map[row['pipeline']]
    shap = load_gt_contributions(row['xai'], int(row['instance']), top_k=10)
    ext  = extractions.get(case_key, {}).get('extraction', {})
    expl = load_explanation_text(prefix, row['xai'], int(row['instance'])) or '(not found)'

    print(f'=== [{label}] {case_key} | RA={row.RA:.2f}, SA={row.SA:.2f} | Primary: {row.primary} ===')
    print(f'Note: {row.note}')
    print()
    print('SHAP Ground Truth (top 4):')
    for c in shap[:4]:
        sign_str = '+' if c['contribution'] > 0 else '-'
        print(f'  rank ? | {c["feature"]:20s} | {sign_str} | |contr|={abs(c["contribution"]):.3f}')
    print()
    print('Extractor Output:')
    sorted_ext = sorted(ext.items(), key=lambda x: x[1].get('rank', 99))
    for feat, info in sorted_ext[:4]:
        print(f'  rank {info.get("rank")} | {feat:20s} | sign={info.get("sign")} | val={info.get("value")}')
    print()
    print('Explanation (first 300 chars):')
    print(expl[:300].replace('\n', ' ') + '...')
    print()

show_case('05_xgb_inst1041', 'E3: cross instance hallucination')
show_case('05_xgb_inst1481', 'E2: sign inversion')
show_case('06_xgb_inst224',  'E1: feature swap')
show_case('05_xgb_inst1677', 'B1: positive/negative grouping')
show_case('05_xgb_inst580',  'B2: close contribution rank swap')
show_case('04_ebm_inst1041', 'C:  yr sign error')

## 5. Key findings

Eval artifacts dominate: 19 of 30 cases (63 percent) are errors of the NB 06 extractor,
not of the XAI pipeline. This means the measured RA/SA values understate the real
explanation quality.

Most frequent eval errors:
* E1 (feature swap, 6 cases): the extractor assigns the wrong feature as rank 0,
  especially temp to hr (instance 224 in all pipelines) and yr to hr (instance 580).
* E2 (sign inversion, 6 cases): the extractor flips the sign, systematically for hr in
  instances 1481 and 3847 (RA correct, SA=0).
* E3 (cross instance, 4 cases): the extractor hallucinates context from another instance,
  especially pronounced for instance 1041 (all pipelines).

Explanation errors (11 cases, 37 percent):
* C (yr sign error, 5 cases): LLMs wrongly present yr=0 (year 2011, negative SHAP
  contribution) as positive "2012 growth". Systematic across all pipelines for EBM
  instance 1041 and instance 2510. Prompt fix candidate: explicit instruction to keep
  the SHAP sign.
* B1 (positive/negative grouping, 2 cases): the explanation separates drivers from
  brakes instead of sorting by absolute contribution, which loses RA when absolute
  contributions are nearly equal.
* B2 (close contribution swap, 4 cases): features with similar absolute contribution in
  the wrong order. Differences often below 10 percent, hard to fix by prompt tuning.

Action for phase 3 (next step): prompt fix for the yr sign error (C), the dominant
explanation error class. Concretely: an explicit instruction to take the SHAP sign from
the JSON input and not infer it from the year context. The eval artifacts (E1 to E3)
must be addressed in parallel by hardening the NB 06 extractor.

## 6. Extraction validity: all 60 LLM cases

The taxonomy (sections 1 to 5) covers only the 30 worst cases. Here the NB 06 extractor
is checked systematically on all 60 LLM cases. Core question: how high is the extractor
rank 0 accuracy, and how much of the measured RA/SA loss is a measurement artefact vs a
real explanation error?

In [ ]:
# 6.1 Load all 60 cases and compare SHAP rank 0 vs extractor rank 0
shap_data = {}
for fname in os.listdir(os.path.join(BASE, 'explanations')):
    if not fname.startswith('local_') or not fname.endswith('.json'):
        continue
    with open(os.path.join(BASE, 'explanations', fname)) as f:
        d = json.load(f)
    shap_data[(d['model'], d['instance_id'])] = sorted(
        d['contributions'], key=lambda x: abs(x['contribution']), reverse=True)

with open(os.path.join(BASE, 'results', 'eval06_ichmoukhamedov', 'extractions.json')) as f:
    extr_all = json.load(f)

df_faith = pd.read_csv(os.path.join(BASE, 'results', 'eval06_ichmoukhamedov', 'faithfulness_metrics.csv'))
df_llm_all = df_faith[df_faith['pipeline_label'] != 'Template'].copy()
prefix_map2 = {'JSON to Text': '04', 'Vision': '05', 'Tool Use': '06'}

records = []
for _, row in df_llm_all.iterrows():
    model  = row['xai_model'].lower()
    inst   = int(row['instance_id'])
    prefix = prefix_map2[row['pipeline_label']]
    key    = f'{prefix}_{model}_inst{inst}'
    shap_ranked = shap_data.get((model, inst), [])
    if not shap_ranked:
        continue
    ext = extr_all.get(key, {}).get('extraction', {})
    # Rank 0 fidelity (feature + sign) via utils.faithfulness.rank0_correctness
    r0 = rank0_correctness(ext, shap_ranked)
    records.append(dict(
        case_key=key, pipeline=row['pipeline_label'],
        xai=row['xai_model'], instance=inst,
        RA=row['RA'], SA=row['SA'], VA=row['VA'],
        gt_r0_feat=r0['gt_r0_feat'], gt_r0_sign=r0['gt_r0_sign'],
        gt_r0_contrib=round(abs(shap_ranked[0]['contribution']), 3),
        ext_r0_feat=r0['ext_r0_feat'], ext_r0_sign=r0['ext_r0_sign'],
        feat_match=r0['feat_match'], sign_match=r0['sign_match'], r0_correct=r0['r0_correct'],
    ))

df_all60 = pd.DataFrame(records)
n_total   = len(df_all60)
n_correct = int(df_all60['r0_correct'].sum())
n_wrong   = n_total - n_correct

print(f'Total LLM cases: {n_total}')
print(f'Extractor rank 0 correct (feature + sign): {n_correct}/{n_total} = {n_correct/n_total*100:.0f}%')
print(f'Rank 0 errors total: {n_wrong}/{n_total} = {n_wrong/n_total*100:.0f}%')
print(f'  Feature mismatch: {int((~df_all60["feat_match"]).sum())}')
print(f'  Sign only error:  {int((df_all60["feat_match"] & ~df_all60["sign_match"]).sum())}')

In [ ]:
# 6.2 Classify all 27 discordant rank 0 cases
# All errors are instance specific: the same mistake in all 3 pipelines for the same (instance, model).
DISCORDANT_CLASS = {
    # inst224 XGB (E1: temp to hr swap), measurement
    '04_xgb_inst224': ('Measurement', 'E1'), '05_xgb_inst224': ('Measurement', 'E1'),
    '06_xgb_inst224': ('Measurement', 'E1'),
    # inst1041 XGB (E3: cross instance hallucination), measurement
    '04_xgb_inst1041': ('Measurement', 'E3'), '05_xgb_inst1041': ('Measurement', 'E3'),
    '06_xgb_inst1041': ('Measurement', 'E3'),
    # inst1481 XGB (E2: sign inversion), measurement
    '04_xgb_inst1481': ('Measurement', 'E2'), '05_xgb_inst1481': ('Measurement', 'E2'),
    '06_xgb_inst1481': ('Measurement', 'E2'),
    # inst1481 EBM (E2/E3), measurement
    '04_ebm_inst1481': ('Measurement', 'E2'), '05_ebm_inst1481': ('Measurement', 'E3'),
    '06_ebm_inst1481': ('Measurement', 'E2'),
    # inst1677 XGB (B1: pos/neg grouping), explanation
    '04_xgb_inst1677': ('Explanation', 'B1'), '05_xgb_inst1677': ('Explanation', 'B1'),
    '06_xgb_inst1677': ('Explanation', 'B1'),
    # inst3543 XGB (E1: yr to hr swap), measurement
    '04_xgb_inst3543': ('Measurement', 'E1'), '05_xgb_inst3543': ('Measurement', 'E1'),
    '06_xgb_inst3543': ('Measurement', 'E1'),
    # inst3847 XGB (E2: sign inversion), measurement
    '04_xgb_inst3847': ('Measurement', 'E2'), '05_xgb_inst3847': ('Measurement', 'E2'),
    '06_xgb_inst3847': ('Measurement', 'E2'),
    # inst3847 EBM (E2: sign inversion), measurement
    '04_ebm_inst3847': ('Measurement', 'E2'), '05_ebm_inst3847': ('Measurement', 'E2'),
    '06_ebm_inst3847': ('Measurement', 'E2'),
    # inst580 EBM (E1: yr to hr swap), measurement
    '04_ebm_inst580':  ('Measurement', 'E1'), '05_ebm_inst580':  ('Measurement', 'E1'),
    '06_ebm_inst580':  ('Measurement', 'E1'),
}

df_discord = df_all60[~df_all60['r0_correct']].copy()
df_discord['error_source'] = df_discord['case_key'].map(
    lambda k: DISCORDANT_CLASS.get(k, ('Unknown', '?'))[0])
df_discord['error_cat'] = df_discord['case_key'].map(
    lambda k: DISCORDANT_CLASS.get(k, ('Unknown', '?'))[1])

print('Classification of all 27 discordant cases:')
print(df_discord[['case_key','RA','SA','gt_r0_feat','gt_r0_sign','ext_r0_feat','ext_r0_sign',
                   'error_source','error_cat']].to_string(index=False))
print()
src_counts = df_discord['error_source'].value_counts()
print('Error source (27 rank 0 errors):')
for src, cnt in src_counts.items():
    print(f'  {src}: {cnt}/{n_wrong} = {cnt/n_wrong*100:.0f}%')

In [ ]:
# 6.3 Spot check 5 concordant cases (r0_correct=True)
spot_keys = ['04_xgb_inst2058', '04_ebm_inst2510', '06_xgb_inst4454',
             '04_xgb_inst2510', '06_xgb_inst2058']


def get_explanation_text(case_key):
    """case_key '04_xgb_inst2058' to explanation text via utils.load_explanation_text."""
    prefix, rest = case_key[:2], case_key[3:]
    model, inst_str = rest.split('_inst')
    return load_explanation_text(prefix, model, int(inst_str)) or '(file not found)'

for key in spot_keys:
    row = df_all60[df_all60['case_key'] == key]
    if row.empty:
        print(f'{key}: not in dataset'); continue
    row = row.iloc[0]
    expl = get_explanation_text(key).replace('\n', ' ')
    print(f'--- {key} | RA={row.RA:.2f} SA={row.SA:.2f}'
          f' | GT-r0={row.gt_r0_feat}({row.gt_r0_sign:+d})'
          f' | EXT-r0={row.ext_r0_feat}({row.ext_r0_sign:+d}) ---')
    print(' ', expl[:260])
    print()

In [ ]:
# Corrected RA/SA upper bound
# For each measurement error case: assume rank 0 would have been correct.
# Correction formula (obs x n + 1)/n, capped at 1.0, see utils.faithfulness.correct_metric.

messung_keys = set(df_discord[df_discord['error_source'] == 'Measurement']['case_key'])


def _n_extracted(case_key):
    return len(extr_all.get(case_key, {}).get('extraction', {}))

df_corr = df_all60.copy()
df_corr['RA_corr'] = df_corr.apply(
    lambda r: correct_metric(r['RA'], _n_extracted(r['case_key']))
              if r['case_key'] in messung_keys and not r['r0_correct']
              else r['RA'], axis=1)
df_corr['SA_corr'] = df_corr.apply(
    lambda r: correct_metric(r['SA'], _n_extracted(r['case_key']))
              if r['case_key'] in messung_keys and not r['r0_correct']
              else r['SA'], axis=1)

print('Observed vs corrected (upper bound) per pipeline:')
grp = df_corr.groupby('pipeline')[['RA','SA','VA','RA_corr','SA_corr']].mean().round(3)
grp['RA_delta'] = (grp['RA_corr'] - grp['RA']).round(3)
grp['SA_delta'] = (grp['SA_corr'] - grp['SA']).round(3)
print(grp.to_string())
print()
print('Total (all 60 LLM cases):')
for col in ['RA','SA','VA','RA_corr','SA_corr']:
    print(f'  {col}: {df_corr[col].mean():.3f}')
ra_gain = (df_corr['RA_corr'] - df_corr['RA']).mean()
sa_gain = (df_corr['SA_corr'] - df_corr['SA']).mean()
print(f'  RA gain from measurement correction: +{ra_gain:.3f}')
print(f'  SA gain from measurement correction: +{sa_gain:.3f}')

In [ ]:
# Save outputs
out_dir = os.path.join(BASE, 'results', 'error_taxonomy')
df_all60.to_csv(os.path.join(out_dir, 'extractor_validity_all60.csv'), index=False)

validity_summary = dict(
    n_total=int(n_total),
    n_r0_correct=int(n_correct),
    n_r0_wrong=int(n_wrong),
    r0_accuracy_pct=round(n_correct/n_total*100, 1),
    n_measurement_error=int((df_discord['error_source'] == 'Measurement').sum()),
    n_explanation_error=int((df_discord['error_source'] == 'Explanation').sum()),
    meas_share_of_r0_errors_pct=round(
        (df_discord['error_source']=='Measurement').sum() / n_wrong * 100, 1),
    RA_observed=round(float(df_corr['RA'].mean()), 3),
    RA_corrected_ub=round(float(df_corr['RA_corr'].mean()), 3),
    SA_observed=round(float(df_corr['SA'].mean()), 3),
    SA_corrected_ub=round(float(df_corr['SA_corr'].mean()), 3),
)
with open(os.path.join(out_dir, 'extractor_validity_summary.json'), 'w') as f:
    json.dump(validity_summary, f, indent=2)
print('Saved extractor_validity_all60.csv and extractor_validity_summary.json')
print(json.dumps(validity_summary, indent=2))

In [ ]:
# Plot: observed vs corrected RA/SA per pipeline
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
pipelines = ['JSON to Text', 'Vision', 'Tool Use']
x = np.arange(len(pipelines))
width = 0.28

for ax, metric, metric_corr, title in [
    (axes[0], 'RA', 'RA_corr', 'Rank Agreement (RA)'),
    (axes[1], 'SA', 'SA_corr', 'Sign Agreement (SA)'),
]:
    obs  = [df_corr[df_corr['pipeline']==p][metric].mean() for p in pipelines]
    corr = [df_corr[df_corr['pipeline']==p][metric_corr].mean() for p in pipelines]
    b1 = ax.bar(x - width/2, obs,  width, label='Observed',          color='#4878cf', alpha=0.85)
    b2 = ax.bar(x + width/2, corr, width, label='Corrected (upper)', color='#6acc65', alpha=0.85)
    for bar, val in zip(list(b1)+list(b2), obs+corr):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                f'{val:.2f}', ha='center', va='bottom', fontsize=9)
    ax.set_xticks(x); ax.set_xticklabels(pipelines, fontsize=10)
    ax.set_ylim(0, 1.05)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.set_ylabel('Mean (0 to 1)', fontsize=10)
    ax.axhline(1.0, color='gray', lw=0.8, ls='--')

plt.suptitle('Observed vs corrected metric (extractor measurement error removed)',
             fontsize=12, y=1.02)
plt.tight_layout()
fig_path = os.path.join(out_dir, 'extractor_validity_corrected.png')
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
display(fig)
print('Saved:', fig_path)

## 7. Consequences for the metric

| Metric | Value |
|--------|-------|
| Extractor rank 0 accuracy (feature + sign) | 55 percent (33/60) |
| Rank 0 errors total | 45 percent (27/60) |
| of which measurement (extractor) at fault | 89 percent (24/27) |
| of which explanation at fault (B1) | 11 percent (3/27) |

Pipeline comparisons stay valid: the extractor errors are instance specific. All three
pipelines suffer the same error on the same instance. This means rank correlations
between pipelines (for example Kendall tau) are not distorted by the bias. The reported
pipeline ranking is valid.

Absolute RA/SA values are understated: the measurement corrected upper bounds are about
+0.12 RA and +0.11 SA above the observed values. To report in the paper: the measured
RA/SA understate the real explanation quality due to extractor artefacts, and corrected
upper bounds are given in the validation table.

Measures (priority):
* (1) Short term: extend the extractor prompt in NB 06 with an explicit sign instruction
  (read the sign from the numeric SHAP value, not from the narrative).
* (2) Medium term: re extract the 7 problematic (instance, model) combinations and update
  the metrics, then observed is approximately equal to corrected.
* (3) Before phase 3b (scaling): an extractor regression as a required test (checks
  feature identity and sign against the SHAP ground truth on example fixtures).